# S5 targeted head and weight-decay controls

This is a minimal update of the existing S5 sweep. The dataset, model body, optimizer, logging, checkpoint format, and swap diagnostic are preserved. The only architectural change is that output-head geometry is now controlled independently from residual geometry.

The notebook runs only these two new controls:

- `layernorm_nowd_control` with LayerNorm residuals, raw unembedding, and WD=0
- `layernorm_bounded_wd1_control` with LayerNorm residuals, normalized fixed-temperature unembedding, and WD=1

Outputs use unique config names and separate aggregate CSV files, so existing S5 results are not overwritten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import math
import json
import random
import itertools

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm


SAVE_DIR = "/content/drive/MyDrive/Grokking_S5_Sweep_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

# BASE CONFIGURATION
N_DEGREE = 5
GROUP_SIZE = math.factorial(N_DEGREE)
FRAC_TRAIN = 0.30
NUM_HEADS = 4
LOG_EVERY = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Notes:
# - Learned-attention positive control: LayerNorm, 3 seeds.
# - Corrected spherical WD comparison: identical normalized unembedding + fixed tau.
# - Robustness sweeps: no-WD spherical, 3 seeds.
# - Uniform-attention validation follows the original ablation implementation:
#   attention scores are fixed to zero, yielding [1/3, 1/3, 1/3].
# - S5 uses AdamW betas=(0.9, 0.999), including uniform-attention runs.
REFERENCE_CONFIGS = [
    # Positive control for the refactored pipeline.
    {"name": "layernorm_learned_control", "model_type": "layernorm",
     "uniform_attention": False, "wd": 1.0, "lr": 1e-3, "tau": None,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    # Main corrected WD comparison.
    {"name": "sphere_wd1_corrected", "model_type": "spherical",
     "uniform_attention": False, "wd": 1.0, "lr": 1e-3, "tau": 10.0,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": list(range(1, 11))},

    {"name": "sphere_nowd_main", "model_type": "spherical",
     "uniform_attention": False, "wd": 0.0, "lr": 1e-3, "tau": 10.0,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    # Learning-rate robustness.
    {"name": "sphere_nowd_lr_3e-4", "model_type": "spherical",
     "uniform_attention": False, "wd": 0.0, "lr": 3e-4, "tau": 10.0,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    {"name": "sphere_nowd_lr_3e-3", "model_type": "spherical",
     "uniform_attention": False, "wd": 0.0, "lr": 3e-3, "tau": 10.0,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    # Temperature robustness.
    {"name": "sphere_nowd_tau_5", "model_type": "spherical",
     "uniform_attention": False, "wd": 0.0, "lr": 1e-3, "tau": 5.0,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    {"name": "sphere_nowd_tau_20", "model_type": "spherical",
     "uniform_attention": False, "wd": 0.0, "lr": 1e-3, "tau": 20.0,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    # Capacity robustness.
    {"name": "sphere_nowd_wide_256", "model_type": "spherical",
     "uniform_attention": False, "wd": 0.0, "lr": 1e-3, "tau": 10.0,
     "d_model": 256, "mlp_dim": 1024, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    {"name": "sphere_nowd_deep_2layer", "model_type": "spherical",
     "uniform_attention": False, "wd": 0.0, "lr": 1e-3, "tau": 10.0,
     "d_model": 128, "mlp_dim": 512, "layers": 2,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    # Uniform-attention validation requested by reviewers.
    {"name": "layernorm_uniform_control", "model_type": "layernorm",
     "uniform_attention": True, "wd": 1.0, "lr": 1e-3, "tau": None,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    {"name": "layernorm_uniform_no_pos", "model_type": "layernorm",
     "uniform_attention": True, "use_positional_embedding": False,
     "wd": 1.0, "lr": 1e-3, "tau": None,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    {"name": "sphere_nowd_uniform", "model_type": "spherical",
     "uniform_attention": True, "wd": 0.0, "lr": 1e-3, "tau": 10.0,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": [1, 2, 3]},

    {"name": "sphere_nowd_long_200k", "model_type": "spherical",
     "uniform_attention": False, "wd": 0.0, "lr": 1e-3, "tau": 10.0,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 200_000, "seeds": [1, 2, 3]},

    {"name": "sphere_raw_wd1_control", "model_type": "spherical",
     "uniform_attention": False, "bounded_head": False,
     "wd": 1.0, "lr": 1e-3, "tau": None,
     "d_model": 128, "mlp_dim": 512, "layers": 1,
     "max_epochs": 100_000, "seeds": list(range(1, 11))},
]

# TARGETED CONTROLS
NEW_CONTROL_CONFIGS = [
    {
        "name": "layernorm_nowd_control",
        "model_type": "layernorm",
        "bounded_head": False,
        "uniform_attention": False,
        "wd": 0.0,
        "lr": 1e-3,
        "tau": None,
        "d_model": 128,
        "mlp_dim": 512,
        "layers": 1,
        "max_epochs": 100_000,
        "seeds": [1, 2, 3],
    },
    {
        "name": "layernorm_bounded_wd1_control",
        "model_type": "layernorm",
        "bounded_head": True,
        "uniform_attention": False,
        "wd": 1.0,
        "lr": 1e-3,
        "tau": 10.0,
        "d_model": 128,
        "mlp_dim": 512,
        "layers": 1,
        "max_epochs": 100_000,
        "seeds": [1, 2, 3],
    },
    {
        "name": "sphere_raw_wd1_control",
        "model_type": "spherical",
        "bounded_head": False,
        "uniform_attention": False,
        "wd": 1.0,
        "lr": 1e-3,
        "tau": None,
        "d_model": 128,
        "mlp_dim": 512,
        "layers": 1,
        "max_epochs": 100_000,
        "seeds": list(range(1, 11)),
    },
]

CONFIGS = NEW_CONTROL_CONFIGS


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


# S5 DATASET GENERATION
def generate_s5_elements():
    elements = list(itertools.permutations(range(N_DEGREE)))
    elem_to_id = {elem: i for i, elem in enumerate(elements)}
    id_to_elem = {i: elem for i, elem in enumerate(elements)}
    return elements, elem_to_id, id_to_elem


def compose_s5(id1, id2, id_to_elem, elem_to_id):
    p1 = id_to_elem[id1]
    p2 = id_to_elem[id2]
    p_out = tuple(p1[p2[i]] for i in range(N_DEGREE))
    return elem_to_id[p_out]


def make_s5_dataset(frac_train, seed=42):
    set_seed(seed)
    rng = random.Random(seed)

    _, elem_to_id, id_to_elem = generate_s5_elements()
    all_pairs = [(a, b) for a in range(GROUP_SIZE) for b in range(GROUP_SIZE)]
    rng.shuffle(all_pairs)

    n_train = int(len(all_pairs) * frac_train)
    op_token = GROUP_SIZE

    train_pairs = all_pairs[:n_train]
    test_pairs = all_pairs[n_train:]

    train_x = torch.tensor([[a, b, op_token] for a, b in train_pairs], dtype=torch.long)
    train_y = torch.tensor(
        [compose_s5(a, b, id_to_elem, elem_to_id) for a, b in train_pairs],
        dtype=torch.long,
    )
    test_x = torch.tensor([[a, b, op_token] for a, b in test_pairs], dtype=torch.long)
    test_y = torch.tensor(
        [compose_s5(a, b, id_to_elem, elem_to_id) for a, b in test_pairs],
        dtype=torch.long,
    )
    return train_x, train_y, test_x, test_y


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_dim, model_type, uniform_attention):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.model_type = model_type
        self.uniform_attention = uniform_attention

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)

        if model_type == "layernorm":
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)

    def _post_residual_norm(self, x, which):
        if self.model_type == "spherical":
            return F.normalize(x, dim=-1)
        if self.model_type == "layernorm":
            return self.norm1(x) if which == 1 else self.norm2(x)
        raise ValueError(f"Unknown model_type: {self.model_type}")

    def forward(self, h):
        B, L, _ = h.shape

        Q = self.W_Q(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)

        if self.uniform_attention:
            # Matches Final_No_attention_all_models.ipynb:
            # zero scores -> softmax gives exactly uniform routing.
            scores = torch.zeros(
                (B, self.num_heads, L, L),
                device=h.device,
                dtype=h.dtype,
            )
        else:
            scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)

        attn = F.softmax(scores, dim=-1)
        attn_out = self.W_O(
            (attn @ V).transpose(1, 2).contiguous().view(B, L, self.d_model)
        )

        h = self._post_residual_norm(h + attn_out, which=1)
        h = self._post_residual_norm(
            h + self.mlp_out(F.relu(self.mlp_in(h))),
            which=2,
        )
        return h


class UnifiedTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes,
        d_model,
        num_heads,
        mlp_dim,
        layers,
        model_type,
        uniform_attention,
        tau,
        bounded_head=None,
        use_positional_embedding=True,
    ):
        super().__init__()
        self.model_type = model_type
        self.bounded_head = (model_type == "spherical") if bounded_head is None else bool(bounded_head)
        self.tau = tau
        self.use_positional_embedding = use_positional_embedding

        if self.bounded_head and self.tau is None:
            raise ValueError("bounded_head=True requires tau")

        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(3, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                num_heads=num_heads,
                mlp_dim=mlp_dim,
                model_type=model_type,
                uniform_attention=uniform_attention,
            )
            for _ in range(layers)
        ])
        self.unembed = nn.Linear(d_model, num_classes, bias=False)

        if model_type == "layernorm":
            self.input_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        _, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_embed(x)
        if self.use_positional_embedding:
            h = h + self.pos_embed(pos)

        if self.model_type == "spherical":
            h = F.normalize(h, dim=-1)
        elif self.model_type == "layernorm":
            h = self.input_norm(h)
        else:
            raise ValueError(f"Unknown model_type: {self.model_type}")

        for block in self.blocks:
            h = block(h)

        h_final = h[:, 2, :]

        if self.bounded_head:
            w_normalized = F.normalize(self.unembed.weight, dim=1)
            return F.linear(h_final, w_normalized) * float(self.tau)

        return self.unembed(h_final)


def train_model(model, train_x, train_y, test_x, test_y, cfg):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["wd"],
        betas=(0.9, 0.999),
    )
    criterion = nn.CrossEntropyLoss()

    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    train_epoch_95 = None
    train_epoch_99 = None
    grok_epoch_95 = None
    grok_epoch_99 = None
    peak_test_acc = 0.0
    status = "completed"

    history = {
        "epochs": [],
        "train_acc": [],
        "test_acc": [],
        "train_loss": [],
        "test_loss": [],
    }

    pbar = tqdm(range(cfg["max_epochs"]), desc=cfg["name"], leave=False)

    for epoch in pbar:
        model.train()
        logits = model(train_x)
        loss = criterion(logits, train_y)

        if not torch.isfinite(loss):
            status = f"non_finite_loss_epoch_{epoch}"
            tqdm.write(
                f'NON-FINITE LOSS | config={cfg["name"]} epoch={epoch}'
            )
            break

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == cfg["max_epochs"] - 1:
            model.eval()
            with torch.no_grad():
                train_logits = model(train_x)
                test_logits = model(test_x)
                train_loss = criterion(train_logits, train_y).item()
                test_loss = criterion(test_logits, test_y).item()
                train_acc = (train_logits.argmax(-1) == train_y).float().mean().item()
                test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

            if not all(np.isfinite(v) for v in [train_loss, test_loss, train_acc, test_acc]):
                status = f"non_finite_metric_epoch_{epoch}"
                tqdm.write(
                    f'NON-FINITE METRIC | config={cfg["name"]} epoch={epoch}'
                )
                break

            peak_test_acc = max(peak_test_acc, test_acc)
            history["epochs"].append(epoch)
            history["train_acc"].append(train_acc)
            history["test_acc"].append(test_acc)
            history["train_loss"].append(train_loss)
            history["test_loss"].append(test_loss)

            pbar.set_postfix(tr=f"{train_acc:.3f}", te=f"{test_acc:.3f}")

            if train_epoch_95 is None and train_acc >= 0.95:
                train_epoch_95 = epoch
            if train_epoch_99 is None and train_acc >= 0.99:
                train_epoch_99 = epoch

            if grok_epoch_95 is None and test_acc >= 0.95:
                grok_epoch_95 = epoch
                tqdm.write(
                    f'{cfg["name"]}: reached 95% test accuracy at epoch {epoch}'
                )

            if grok_epoch_99 is None and test_acc >= 0.99:
                grok_epoch_99 = epoch
                tqdm.write(
                    f'{cfg["name"]}: reached 99% test accuracy at epoch {epoch}'
                )

    return (
        history,
        train_epoch_95,
        train_epoch_99,
        grok_epoch_95,
        grok_epoch_99,
        peak_test_acc,
        status,
    )



def evaluate_input_swap(model, test_x, test_y):
    """
    Evaluate order sensitivity on S5.

    For each original input [a, b, op], also evaluate [b, a, op].
    The swapped target is recomputed as b o a, not reused from a o b.

    Returns:
      - original_accuracy: accuracy on a o b
      - swapped_accuracy: accuracy on b o a
      - prediction_invariance_rate: fraction where f(a,b) == f(b,a)
      - noncommuting_original_accuracy: accuracy on examples with a o b != b o a
      - noncommuting_swapped_accuracy: swapped accuracy on the same subset
      - noncommuting_prediction_invariance_rate: invariance rate on that subset
      - noncommuting_fraction: fraction of test examples that do not commute
    """
    model.eval()

    # train_model moves only its local tensor references to the GPU.
    # The outer-loop test tensors can therefore still be on CPU.
    model_device = next(model.parameters()).device
    test_x = test_x.to(model_device)
    test_y = test_y.to(model_device)

    test_x_cpu = test_x.detach().cpu()
    a_ids = test_x_cpu[:, 0].tolist()
    b_ids = test_x_cpu[:, 1].tolist()

    _, elem_to_id, id_to_elem = generate_s5_elements()
    swapped_targets = torch.tensor(
        [compose_s5(b, a, id_to_elem, elem_to_id) for a, b in zip(a_ids, b_ids)],
        dtype=torch.long,
        device=model_device,
    )

    swapped_x = test_x.clone()
    swapped_x[:, 0] = test_x[:, 1]
    swapped_x[:, 1] = test_x[:, 0]

    with torch.no_grad():
        original_pred = model(test_x).argmax(dim=-1)
        swapped_pred = model(swapped_x).argmax(dim=-1)

    original_correct = original_pred.eq(test_y)
    swapped_correct = swapped_pred.eq(swapped_targets)
    same_prediction = original_pred.eq(swapped_pred)

    noncommuting_mask = test_y.ne(swapped_targets)
    noncommuting_count = int(noncommuting_mask.sum().item())

    def masked_mean(values, mask):
        if int(mask.sum().item()) == 0:
            return None
        return values[mask].float().mean().item()

    return {
        "original_accuracy": original_correct.float().mean().item(),
        "swapped_accuracy": swapped_correct.float().mean().item(),
        "prediction_invariance_rate": same_prediction.float().mean().item(),
        "noncommuting_original_accuracy": masked_mean(original_correct, noncommuting_mask),
        "noncommuting_swapped_accuracy": masked_mean(swapped_correct, noncommuting_mask),
        "noncommuting_prediction_invariance_rate": masked_mean(same_prediction, noncommuting_mask),
        "noncommuting_fraction": noncommuting_mask.float().mean().item(),
        "noncommuting_count": noncommuting_count,
        "test_count": int(test_y.numel()),
    }


def safe_name(text):
    return "".join(c if c.isalnum() or c in "-_." else "_" for c in text)


summary_rows = []

print("device:", DEVICE)
print("running targeted S5 controls:")
for cfg in CONFIGS:
    head_name = "bounded" if cfg["bounded_head"] else "raw"
    print(
        f"  {cfg['name']} | residual={cfg['model_type']} | "
        f"head={head_name} | wd={cfg['wd']} | seeds={cfg['seeds']}"
    )

for cfg in CONFIGS:
    cfg_dir = os.path.join(SAVE_DIR, safe_name(cfg["name"]))
    os.makedirs(cfg_dir, exist_ok=True)

    for seed in cfg["seeds"]:
        stem = f'{safe_name(cfg["name"])}_seed{seed}'
        log_path = os.path.join(cfg_dir, stem + ".json")
        ckpt_path = os.path.join(cfg_dir, stem + ".pt")

        # Resume at completed model granularity.
        if os.path.exists(log_path) and os.path.exists(ckpt_path):
            with open(log_path, "r") as f:
                saved = json.load(f)
            summary_rows.append(saved["summary"])
            print("skip:", stem)
            continue

        train_x, train_y, test_x, test_y = make_s5_dataset(FRAC_TRAIN, seed)
        set_seed(seed)

        model = UnifiedTransformer(
            vocab_size=GROUP_SIZE + 1,
            num_classes=GROUP_SIZE,
            d_model=cfg["d_model"],
            num_heads=NUM_HEADS,
            mlp_dim=cfg["mlp_dim"],
            layers=cfg["layers"],
            model_type=cfg["model_type"],
            uniform_attention=cfg["uniform_attention"],
            tau=cfg["tau"],
            bounded_head=cfg.get("bounded_head", cfg["model_type"] == "spherical"),
            use_positional_embedding=cfg.get("use_positional_embedding", True),
        ).to(DEVICE)

        (
            history,
            train95,
            train99,
            grok95,
            grok99,
            peak_test_acc,
            status,
        ) = train_model(
            model, train_x, train_y, test_x, test_y, cfg
        )

        try:
            swap_metrics = evaluate_input_swap(
                model=model,
                test_x=test_x,
                test_y=test_y,
            )
        except Exception as exc:
            # Do not lose a completed training run because a post-hoc diagnostic failed.
            tqdm.write(
                f'SWAP EVALUATION FAILED | config={cfg["name"]} seed={seed} | {exc}'
            )
            swap_metrics = {
                "original_accuracy": None,
                "swapped_accuracy": None,
                "prediction_invariance_rate": None,
                "noncommuting_original_accuracy": None,
                "noncommuting_swapped_accuracy": None,
                "noncommuting_prediction_invariance_rate": None,
                "noncommuting_fraction": None,
                "noncommuting_count": None,
                "test_count": int(test_y.numel()),
            }

        if history["epochs"]:
            final_train_acc = history["train_acc"][-1]
            final_test_acc = history["test_acc"][-1]
            final_train_loss = history["train_loss"][-1]
            final_test_loss = history["test_loss"][-1]
            last_logged_epoch = history["epochs"][-1]
        else:
            final_train_acc = None
            final_test_acc = None
            final_train_loss = None
            final_test_loss = None
            last_logged_epoch = None

        summary = {
            "config": cfg["name"],
            "model_type": cfg["model_type"],
            "bounded_head": cfg.get("bounded_head", cfg["model_type"] == "spherical"),
            "uniform_attention": cfg["uniform_attention"],
            "use_positional_embedding": cfg.get("use_positional_embedding", True),
            "seed": seed,
            "wd": cfg["wd"],
            "lr": cfg["lr"],
            "tau": cfg["tau"],
            "d_model": cfg["d_model"],
            "mlp_dim": cfg["mlp_dim"],
            "layers": cfg["layers"],
            "max_epochs": cfg["max_epochs"],
            "last_logged_epoch": last_logged_epoch,
            "status": status,
            "train_epoch_95": train95,
            "train_epoch_99": train99,
            "grok_epoch_95": grok95,
            "grok_epoch_99": grok99,
            "gap_95": None if train95 is None or grok95 is None else grok95 - train95,
            "gap_99": None if train99 is None or grok99 is None else grok99 - train99,
            "peak_test_acc": peak_test_acc,
            "final_train_acc": final_train_acc,
            "final_test_acc": final_test_acc,
            "final_train_loss": final_train_loss,
            "final_test_loss": final_test_loss,
            "swap_original_accuracy": swap_metrics["original_accuracy"],
            "swap_swapped_accuracy": swap_metrics["swapped_accuracy"],
            "swap_prediction_invariance_rate": swap_metrics["prediction_invariance_rate"],
            "swap_noncommuting_original_accuracy": swap_metrics["noncommuting_original_accuracy"],
            "swap_noncommuting_swapped_accuracy": swap_metrics["noncommuting_swapped_accuracy"],
            "swap_noncommuting_prediction_invariance_rate": swap_metrics["noncommuting_prediction_invariance_rate"],
            "swap_noncommuting_fraction": swap_metrics["noncommuting_fraction"],
            "swap_noncommuting_count": swap_metrics["noncommuting_count"],
            "swap_test_count": swap_metrics["test_count"],
        }

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "summary": summary,
                "config": cfg,
            },
            ckpt_path,
        )
        with open(log_path, "w") as f:
            json.dump({"summary": summary, "history": history}, f, indent=2)

        summary_rows.append(summary)
        pd.DataFrame(summary_rows).to_csv(
            os.path.join(SAVE_DIR, "summary_new_head_controls_incremental.csv"),
            index=False,
        )
        print("saved:", stem)

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


df = pd.DataFrame(summary_rows)
df.to_csv(os.path.join(SAVE_DIR, "summary_new_head_controls_final.csv"), index=False)

print("\nPer-run results:")
print(df.sort_values(["config", "seed"]).to_string(index=False))

print("\nConfiguration summary:")
agg = (
    df.groupby("config", dropna=False)
      .agg(
          seeds=("seed", "count"),
          completed=("status", lambda x: (x == "completed").sum()),
          mean_peak_acc=("peak_test_acc", "mean"),
          std_peak_acc=("peak_test_acc", "std"),
          max_peak_acc=("peak_test_acc", "max"),
          mean_final_train_acc=("final_train_acc", "mean"),
          mean_swap_original_acc=("swap_original_accuracy", "mean"),
          mean_swap_swapped_acc=("swap_swapped_accuracy", "mean"),
          mean_prediction_invariance=("swap_prediction_invariance_rate", "mean"),
          mean_noncommuting_swapped_acc=("swap_noncommuting_swapped_accuracy", "mean"),
          mean_noncommuting_invariance=("swap_noncommuting_prediction_invariance_rate", "mean"),
          mean_gap_95=("gap_95", "mean"),
          std_gap_95=("gap_95", "std"),
          mean_gap_99=("gap_99", "mean"),
          std_gap_99=("gap_99", "std"),
          reached_95=("grok_epoch_95", lambda x: x.notna().sum()),
          reached_99=("grok_epoch_99", lambda x: x.notna().sum()),
      )
      .reset_index()
)
print(agg.to_string(index=False))
agg.to_csv(os.path.join(SAVE_DIR, "configuration_new_head_controls_summary.csv"), index=False)

## Post-hoc swapped-input evaluation

This cell evaluates order sensitivity on saved S5 checkpoints without retraining. It compares predictions for `(a,b)` and `(b,a)`, recomputes the correct swapped target `b∘a`, and reports metrics specifically on non-commuting pairs.

In [ ]:

# POST-HOC SWAP EVALUATION FOR ALREADY-SAVED CHECKPOINTS
# Run this cell after the main definitions cell if some checkpoints already exist.
# It updates each JSON summary and rewrites the CSV files without retraining.

posthoc_rows = []

for cfg in CONFIGS:
    cfg_dir = os.path.join(SAVE_DIR, safe_name(cfg["name"]))

    for seed in cfg["seeds"]:
        stem = f'{safe_name(cfg["name"])}_seed{seed}'
        log_path = os.path.join(cfg_dir, stem + ".json")
        ckpt_path = os.path.join(cfg_dir, stem + ".pt")

        if not (os.path.exists(log_path) and os.path.exists(ckpt_path)):
            continue

        train_x, train_y, test_x, test_y = make_s5_dataset(FRAC_TRAIN, seed)
        train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
        test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

        model = UnifiedTransformer(
            vocab_size=GROUP_SIZE + 1,
            num_classes=GROUP_SIZE,
            d_model=cfg["d_model"],
            num_heads=NUM_HEADS,
            mlp_dim=cfg["mlp_dim"],
            layers=cfg["layers"],
            model_type=cfg["model_type"],
            uniform_attention=cfg["uniform_attention"],
            tau=cfg["tau"],
            bounded_head=cfg.get("bounded_head", cfg["model_type"] == "spherical"),
            use_positional_embedding=cfg.get("use_positional_embedding", True),
        ).to(DEVICE)

        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])

        swap_metrics = evaluate_input_swap(model, test_x, test_y)

        with open(log_path, "r") as f:
            saved = json.load(f)

        saved["summary"].update({
            "swap_original_accuracy": swap_metrics["original_accuracy"],
            "swap_swapped_accuracy": swap_metrics["swapped_accuracy"],
            "swap_prediction_invariance_rate": swap_metrics["prediction_invariance_rate"],
            "swap_noncommuting_original_accuracy": swap_metrics["noncommuting_original_accuracy"],
            "swap_noncommuting_swapped_accuracy": swap_metrics["noncommuting_swapped_accuracy"],
            "swap_noncommuting_prediction_invariance_rate": swap_metrics["noncommuting_prediction_invariance_rate"],
            "swap_noncommuting_fraction": swap_metrics["noncommuting_fraction"],
            "swap_noncommuting_count": swap_metrics["noncommuting_count"],
            "swap_test_count": swap_metrics["test_count"],
        })

        with open(log_path, "w") as f:
            json.dump(saved, f, indent=2)

        checkpoint["summary"] = saved["summary"]
        torch.save(checkpoint, ckpt_path)

        posthoc_rows.append(saved["summary"])
        print(
            f'updated {stem} | '
            f'orig={swap_metrics["original_accuracy"]:.4f} | '
            f'swapped={swap_metrics["swapped_accuracy"]:.4f} | '
            f'noncomm_inv={swap_metrics["noncommuting_prediction_invariance_rate"]:.4f}'
        )

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if posthoc_rows:
    posthoc_df = pd.DataFrame(posthoc_rows)
    posthoc_df.to_csv(os.path.join(SAVE_DIR, "summary_final_with_swap.csv"), index=False)

    posthoc_agg = (
        posthoc_df.groupby("config", dropna=False)
        .agg(
            seeds=("seed", "count"),
            mean_original_acc=("swap_original_accuracy", "mean"),
            mean_swapped_acc=("swap_swapped_accuracy", "mean"),
            mean_prediction_invariance=("swap_prediction_invariance_rate", "mean"),
            mean_noncommuting_original_acc=("swap_noncommuting_original_accuracy", "mean"),
            mean_noncommuting_swapped_acc=("swap_noncommuting_swapped_accuracy", "mean"),
            mean_noncommuting_invariance=("swap_noncommuting_prediction_invariance_rate", "mean"),
        )
        .reset_index()
    )
    posthoc_agg.to_csv(os.path.join(SAVE_DIR, "swap_configuration_new_head_controls_summary.csv"), index=False)
    print("\\nSwap summary:")
    print(posthoc_agg.to_string(index=False))
else:
    print("No completed checkpoints found for post-hoc swap evaluation.")


In [ ]:
# OPTIONAL: compact summary of the two new controls
# Run after the training cell. This reads saved JSON summaries only.

import glob

rows = []
for cfg in CONFIGS:
    cfg_dir = os.path.join(SAVE_DIR, safe_name(cfg["name"]))
    for path in sorted(glob.glob(os.path.join(cfg_dir, "*.json"))):
        with open(path, "r") as f:
            saved = json.load(f)
        if saved.get("summary"):
            rows.append(saved["summary"])

if not rows:
    print("No completed new-control summaries found yet.")
else:
    new_df = pd.DataFrame(rows).sort_values(["config", "seed"])
    columns = [
        "config", "seed", "model_type", "bounded_head", "wd",
        "train_epoch_95", "grok_epoch_95", "gap_95",
        "train_epoch_99", "grok_epoch_99", "gap_99",
        "peak_test_acc", "status",
    ]
    print("\nPer-seed new controls:")
    print(new_df[columns].to_string(index=False))

    compact = (
        new_df.groupby("config", dropna=False)
        .agg(
            seeds=("seed", "count"),
            reached_95=("grok_epoch_95", lambda x: x.notna().sum()),
            reached_99=("grok_epoch_99", lambda x: x.notna().sum()),
            mean_test_95=("grok_epoch_95", "mean"),
            std_test_95=("grok_epoch_95", "std"),
            mean_test_99=("grok_epoch_99", "mean"),
            std_test_99=("grok_epoch_99", "std"),
            mean_gap_95=("gap_95", "mean"),
            mean_gap_99=("gap_99", "mean"),
            mean_peak_test_acc=("peak_test_acc", "mean"),
        )
        .reset_index()
    )
    print("\nConfiguration summary:")
    print(compact.to_string(index=False))


In [ ]:
# ============================================================
# STANDALONE S5 MASTER SUMMARY
# Reads every saved S5 JSON under SAVE_DIR and combines old/new runs.
# Safe to run without retraining anything.
# ============================================================

import os
import json
import glob
import numpy as np
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/Grokking_S5_Sweep_v2"

# ------------------------------------------------------------
# 1. Load every per-run JSON recursively
# ------------------------------------------------------------

json_paths = glob.glob(
    os.path.join(SAVE_DIR, "**", "*.json"),
    recursive=True,
)

rows = []
load_errors = []

for path in json_paths:
    try:
        with open(path, "r") as f:
            payload = json.load(f)

        # Expected format:
        # {"summary": {...}, "history": {...}}
        if isinstance(payload, dict) and isinstance(payload.get("summary"), dict):
            row = dict(payload["summary"])
            row["_source_json"] = path
            rows.append(row)

    except Exception as exc:
        load_errors.append((path, str(exc)))

if not rows:
    raise RuntimeError(
        f"No per-run JSON summaries found under:\n{SAVE_DIR}"
    )

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 2. Normalize old/new column names
# ------------------------------------------------------------

rename_candidates = {
    "Architecture": "config",
    "architecture": "config",
    "Seed": "seed",
    "Grok Epoch": "grok_epoch_99",
    "grok_epoch": "grok_epoch_99",
    "Peak Test Acc": "peak_test_acc",
    "peak_accuracy": "peak_test_acc",
}

for old, new in rename_candidates.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old: new})

if "config" not in df.columns or "seed" not in df.columns:
    print("Available columns:")
    print(df.columns.tolist())
    raise KeyError("Could not identify config and seed columns.")

# Ensure seed is numeric where possible.
df["seed"] = pd.to_numeric(df["seed"], errors="coerce")

# Old files may not have status.
if "status" not in df.columns:
    df["status"] = "completed"
else:
    df["status"] = df["status"].fillna("completed")

# Add missing columns so aggregation never crashes.
required_columns = [
    "model_type",
    "bounded_head",
    "uniform_attention",
    "use_positional_embedding",
    "wd",
    "lr",
    "tau",
    "d_model",
    "mlp_dim",
    "layers",
    "max_epochs",
    "last_logged_epoch",
    "train_epoch_95",
    "train_epoch_99",
    "grok_epoch_95",
    "grok_epoch_99",
    "gap_95",
    "gap_99",
    "peak_test_acc",
    "final_train_acc",
    "final_test_acc",
    "final_train_loss",
    "final_test_loss",
    "swap_original_accuracy",
    "swap_swapped_accuracy",
    "swap_prediction_invariance_rate",
    "swap_noncommuting_original_accuracy",
    "swap_noncommuting_swapped_accuracy",
    "swap_noncommuting_prediction_invariance_rate",
]

for col in required_columns:
    if col not in df.columns:
        df[col] = np.nan

# Reconstruct gaps if onset columns exist but gaps are missing.
missing_gap95 = (
    df["gap_95"].isna()
    & df["train_epoch_95"].notna()
    & df["grok_epoch_95"].notna()
)
df.loc[missing_gap95, "gap_95"] = (
    df.loc[missing_gap95, "grok_epoch_95"]
    - df.loc[missing_gap95, "train_epoch_95"]
)

missing_gap99 = (
    df["gap_99"].isna()
    & df["train_epoch_99"].notna()
    & df["grok_epoch_99"].notna()
)
df.loc[missing_gap99, "gap_99"] = (
    df.loc[missing_gap99, "grok_epoch_99"]
    - df.loc[missing_gap99, "train_epoch_99"]
)

# ------------------------------------------------------------
# 3. Deduplicate by config + seed
# ------------------------------------------------------------
# If multiple JSON files exist for the same config/seed, keep the
# most recently modified file.

df["_mtime"] = df["_source_json"].map(os.path.getmtime)

duplicate_counts = (
    df.groupby(["config", "seed"], dropna=False)
      .size()
      .reset_index(name="copies")
)
duplicates = duplicate_counts[duplicate_counts["copies"] > 1]

df = (
    df.sort_values("_mtime")
      .drop_duplicates(subset=["config", "seed"], keep="last")
      .sort_values(["config", "seed"])
      .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Helpers
# ------------------------------------------------------------

def successful_mean(series):
    """Mean over non-missing successful onset values."""
    values = pd.to_numeric(series, errors="coerce").dropna()
    return values.mean() if len(values) else np.nan


def successful_std(series):
    """Sample std over successful onset values."""
    values = pd.to_numeric(series, errors="coerce").dropna()
    return values.std(ddof=1) if len(values) > 1 else np.nan


def numeric_range(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if not len(values):
        return ""
    return f"{values.min():,.0f}-{values.max():,.0f}"


def mean_std_text(mean_value, std_value, digits=0):
    if pd.isna(mean_value):
        return "-"
    if pd.isna(std_value):
        return f"{mean_value:,.{digits}f}"
    return f"{mean_value:,.{digits}f} +/- {std_value:,.{digits}f}"


# ------------------------------------------------------------
# 5. Master per-run table
# ------------------------------------------------------------

per_run_columns = [
    "config",
    "model_type",
    "bounded_head",
    "seed",
    "wd",
    "lr",
    "tau",
    "status",
    "train_epoch_95",
    "train_epoch_99",
    "grok_epoch_95",
    "grok_epoch_99",
    "gap_95",
    "gap_99",
    "peak_test_acc",
    "final_train_acc",
    "final_test_acc",
    "last_logged_epoch",
]

print("=" * 120)
print("ALL S5 PER-RUN RESULTS")
print("=" * 120)
print(
    df[per_run_columns]
    .sort_values(["config", "seed"])
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Full configuration summary
# ------------------------------------------------------------

summary_rows = []

for config, group in df.groupby("config", dropna=False):
    grok95 = pd.to_numeric(group["grok_epoch_95"], errors="coerce")
    grok99 = pd.to_numeric(group["grok_epoch_99"], errors="coerce")
    gap95 = pd.to_numeric(group["gap_95"], errors="coerce")
    gap99 = pd.to_numeric(group["gap_99"], errors="coerce")
    peaks = pd.to_numeric(group["peak_test_acc"], errors="coerce")
    failed99 = group[grok99.isna()]

    summary_rows.append({
        "config": config,
        "model_type": group["model_type"].dropna().iloc[0]
            if group["model_type"].notna().any() else np.nan,
        "bounded_head": group["bounded_head"].dropna().iloc[0]
            if group["bounded_head"].notna().any() else np.nan,
        "wd": group["wd"].dropna().iloc[0]
            if group["wd"].notna().any() else np.nan,
        "n": len(group),
        "completed": int((group["status"] == "completed").sum()),
        "reached_95": int(grok95.notna().sum()),
        "reached_99": int(grok99.notna().sum()),
        "success_99": f"{int(grok99.notna().sum())}/{len(group)}",
        "test95_mean": successful_mean(grok95),
        "test95_std": successful_std(grok95),
        "test99_mean": successful_mean(grok99),
        "test99_std": successful_std(grok99),
        "test99_range": numeric_range(grok99),
        "gap95_mean": successful_mean(gap95),
        "gap95_std": successful_std(gap95),
        "gap99_mean": successful_mean(gap99),
        "gap99_std": successful_std(gap99),
        "mean_peak_acc": peaks.mean(),
        "min_peak_acc": peaks.min(),
        "max_peak_acc": peaks.max(),
        "failed_mean_peak_acc": pd.to_numeric(
            failed99["peak_test_acc"], errors="coerce"
        ).mean(),
        "failed_peak_range": (
            f"{pd.to_numeric(failed99['peak_test_acc'], errors='coerce').min():.4f}-"
            f"{pd.to_numeric(failed99['peak_test_acc'], errors='coerce').max():.4f}"
            if len(failed99) else ""
        ),
        "mean_final_train_acc": pd.to_numeric(
            group["final_train_acc"], errors="coerce"
        ).mean(),
        "mean_final_test_acc": pd.to_numeric(
            group["final_test_acc"], errors="coerce"
        ).mean(),
    })

summary = pd.DataFrame(summary_rows)

summary["Test-95"] = summary.apply(
    lambda r: mean_std_text(r["test95_mean"], r["test95_std"]), axis=1
)
summary["Test-99"] = summary.apply(
    lambda r: mean_std_text(r["test99_mean"], r["test99_std"]), axis=1
)
summary["Gap-95"] = summary.apply(
    lambda r: mean_std_text(r["gap95_mean"], r["gap95_std"]), axis=1
)
summary["Gap-99"] = summary.apply(
    lambda r: mean_std_text(r["gap99_mean"], r["gap99_std"]), axis=1
)

display_columns = [
    "config",
    "model_type",
    "bounded_head",
    "wd",
    "n",
    "success_99",
    "Test-95",
    "Test-99",
    "test99_range",
    "Gap-95",
    "Gap-99",
    "mean_peak_acc",
    "failed_peak_range",
    "mean_final_train_acc",
]

print("\n" + "=" * 120)
print("ALL S5 CONFIGURATION SUMMARY")
print("Onset means and standard deviations use successful seeds only.")
print("=" * 120)
print(
    summary[display_columns]
    .sort_values(["config"])
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. Matched WD=1 residual x head matrix
# ------------------------------------------------------------

wd1_matrix_configs = [
    "layernorm_learned_control",
    "layernorm_bounded_wd1_control",
    "sphere_raw_wd1_control",
    "sphere_wd1_corrected",
]

wd1 = summary[summary["config"].isin(wd1_matrix_configs)].copy()

wd1_order = {
    "layernorm_learned_control": 0,
    "layernorm_bounded_wd1_control": 1,
    "sphere_raw_wd1_control": 2,
    "sphere_wd1_corrected": 3,
}

wd1["order"] = wd1["config"].map(wd1_order)
wd1 = wd1.sort_values("order")

print("\n" + "=" * 120)
print("S5 MATCHED WD=1 RESIDUAL x HEAD MATRIX")
print("=" * 120)

matrix_columns = [
    "config",
    "model_type",
    "bounded_head",
    "n",
    "success_99",
    "Test-99",
    "test99_range",
    "Gap-99",
    "mean_peak_acc",
    "failed_peak_range",
]

print(wd1[matrix_columns].to_string(index=False))

# Compact reviewer-facing matrix.
matrix_labels = {
    "layernorm_learned_control": ("LayerNorm", "raw"),
    "layernorm_bounded_wd1_control": ("LayerNorm", "bounded"),
    "sphere_raw_wd1_control": ("Sphere", "raw"),
    "sphere_wd1_corrected": ("Sphere", "bounded"),
}

review_rows = []

for _, row in wd1.iterrows():
    residual, head = matrix_labels[row["config"]]
    review_rows.append({
        "Residual": residual,
        "Head": head,
        "n": int(row["n"]),
        "Reached 99": row["success_99"],
        "Test-99": row["Test-99"],
        "Range": row["test99_range"] or "-",
        "Gap-99": row["Gap-99"],
        "Mean peak": (
            f"{100 * row['mean_peak_acc']:.2f}%"
            if pd.notna(row["mean_peak_acc"]) else "-"
        ),
    })

review_matrix = pd.DataFrame(review_rows)

print("\nReviewer-facing matrix:")
print(review_matrix.to_string(index=False))

# ------------------------------------------------------------
# 8. WD=0 diagnostic panel
# ------------------------------------------------------------

wd0_names = [
    "layernorm_nowd_control",
    "sphere_nowd_main",
    "sphere_nowd_lr_3e-4",
    "sphere_nowd_lr_3e-3",
    "sphere_nowd_tau_5",
    "sphere_nowd_tau_20",
    "sphere_nowd_wide_256",
    "sphere_nowd_deep_2layer",
    "sphere_nowd_uniform",
    "sphere_nowd_long_200k",
]

wd0 = summary[summary["config"].isin(wd0_names)].copy()

print("\n" + "=" * 120)
print("S5 NO-WEIGHT-DECAY DIAGNOSTICS")
print("=" * 120)

print(
    wd0[
        [
            "config",
            "n",
            "success_99",
            "mean_peak_acc",
            "failed_peak_range",
            "mean_final_train_acc",
        ]
    ]
    .sort_values("config")
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. Swap summaries already stored inside training JSONs
# ------------------------------------------------------------

swap_columns = [
    "swap_original_accuracy",
    "swap_swapped_accuracy",
    "swap_prediction_invariance_rate",
    "swap_noncommuting_original_accuracy",
    "swap_noncommuting_swapped_accuracy",
    "swap_noncommuting_prediction_invariance_rate",
]

if df[swap_columns].notna().any().any():
    swap_summary = (
        df.groupby("config", dropna=False)
          .agg(
              seeds=("seed", "count"),
              mean_original_acc=("swap_original_accuracy", "mean"),
              mean_swapped_acc=("swap_swapped_accuracy", "mean"),
              mean_prediction_invariance=(
                  "swap_prediction_invariance_rate", "mean"
              ),
              mean_noncommuting_original_acc=(
                  "swap_noncommuting_original_accuracy", "mean"
              ),
              mean_noncommuting_swapped_acc=(
                  "swap_noncommuting_swapped_accuracy", "mean"
              ),
              mean_noncommuting_invariance=(
                  "swap_noncommuting_prediction_invariance_rate", "mean"
              ),
          )
          .reset_index()
    )

    print("\n" + "=" * 120)
    print("ALL AVAILABLE STANDARD SWAP METRICS")
    print("=" * 120)
    print(swap_summary.to_string(index=False))

# ------------------------------------------------------------
# 10. Find and print post-hoc dual-held-out CSVs if present
# ------------------------------------------------------------

csv_paths = glob.glob(
    os.path.join(SAVE_DIR, "**", "*.csv"),
    recursive=True,
)

dual_candidates = [
    p for p in csv_paths
    if "dual" in os.path.basename(p).lower()
    or "heldout" in os.path.basename(p).lower()
]

if dual_candidates:
    print("\n" + "=" * 120)
    print("FOUND DUAL-HELD-OUT / SWAP CSV FILES")
    print("=" * 120)

    for path in sorted(dual_candidates):
        print(path)

        try:
            dual_df = pd.read_csv(path)
            print(dual_df.to_string(index=False))
        except Exception as exc:
            print("Could not read:", exc)
else:
    print("\nNo dual-held-out CSV was found under SAVE_DIR.")
    print("Run the post-hoc dual-held-out evaluator, then rerun this cell.")

# ------------------------------------------------------------
# 11. Duplicate and load diagnostics
# ------------------------------------------------------------

if len(duplicates):
    print("\n" + "=" * 120)
    print("DUPLICATE CONFIG-SEED FILES FOUND")
    print("The newest JSON was retained for each config-seed pair.")
    print("=" * 120)
    print(duplicates.to_string(index=False))

if load_errors:
    print("\nJSON files that could not be loaded:")
    for path, error in load_errors:
        print(f"- {path}: {error}")

# ------------------------------------------------------------
# 12. Save unified master CSVs
# ------------------------------------------------------------

master_per_run_path = os.path.join(
    SAVE_DIR,
    "S5_master_all_runs.csv",
)
master_summary_path = os.path.join(
    SAVE_DIR,
    "S5_master_all_config_summary.csv",
)
master_matrix_path = os.path.join(
    SAVE_DIR,
    "S5_master_wd1_2x2_matrix.csv",
)

df.drop(columns=["_mtime"], errors="ignore").to_csv(
    master_per_run_path,
    index=False,
)
summary.to_csv(
    master_summary_path,
    index=False,
)
review_matrix.to_csv(
    master_matrix_path,
    index=False,
)

print("\n" + "=" * 120)
print("SAVED")
print("=" * 120)
print(master_per_run_path)
print(master_summary_path)
print(master_matrix_path)